# MMAction interaction v1 → đánh nhau / vật nhau / ngã chung

Notebook này chạy **trên Colab GPU**, fine-tune `InteractionHead` từ checkpoint v1 và giữ nguyên AVA backbone + person detector. Đầu ra là `interaction.pt`, `interaction.json`, báo cáo và ZIP trong Google Drive.

**Giới hạn:** SCFD có nhãn fight chung và tác giả cho biết gồm cả wrestling, nhưng không có nhãn riêng từng clip vật nhau. Notebook sẽ đo đánh nhau và ngã; chưa thể chứng minh recall riêng cho vật nhau. Đừng thay checkpoint đang dùng trước khi xem báo cáo và thử video của bạn.

Chọn Runtime → GPU. Chạy tuần tự các cell. Cell tải GMDCSA24 khoảng 1,1 GB và SCFD khoảng 91 MB; cell trích đặc trưng có thể chạy lâu nhưng lưu cache trên Drive.

In [ ]:
from pathlib import Path
import base64, gzip, hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
from google.colab import drive, files

# Thay RUN_TAG để tạo một lần train mới; không ghi đè checkpoint cũ.
RUN_TAG = 'fall_wrestle_pilot_01'
EPOCHS = 40
TRAIN_MINUTES = 90
assert RUN_TAG and all(c.isalnum() or c in '_-' for c in RUN_TAG)
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive', timeout_ms=180000)
DRIVE = Path('/content/drive/MyDrive/SENTINEL')
RUN = DRIVE / 'artifacts/mmaction_interaction_fall_wrestle' / RUN_TAG
RUN.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/general_fall_v1'); WORK.mkdir(exist_ok=True)
CODE = WORK / 'code'; CODE.mkdir(exist_ok=True)
DEMO = CODE / 'mmaction2_demo'
WEIGHTS = WORK / 'weights'; WEIGHTS.mkdir(exist_ok=True)
CACHE = RUN / 'feature_cache'
print('Output on Drive:', RUN)


In [ ]:
# Code gốc cố định tại commit đã dùng cho v1, module fine-tune được nhúng trong notebook.
SOURCE_REVISION = '75defa41acfa358b1618822136577c6af182a6a5'
source_zip = WORK / 'source.zip'
if not (DEMO / 'action_engine.py').is_file():
    url = 'https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/' + SOURCE_REVISION
    with urllib.request.urlopen(url, timeout=180) as response, source_zip.open('wb') as target:
        shutil.copyfileobj(response, target)
    with zipfile.ZipFile(source_zip) as package:
        for item in package.infolist():
            target = (WORK / item.filename).resolve()
            assert target.is_relative_to(WORK.resolve()), item.filename
        package.extractall(WORK)
    source = WORK / ('anomaly-behavior-detector-system-' + SOURCE_REVISION)
    shutil.copytree(source, CODE, dirs_exist_ok=True)
DEMO.mkdir(exist_ok=True)
(DEMO / 'train_general_fall_v1.py').write_bytes(gzip.decompress(base64.b64decode('H4sIAAAAAAAC/7U7a2/bSJLf9Sv6eDiATGRGUvzUjAbnc5ybALsZI/HMAicIRItsWVxTJJcPO47h/3716Cab1MPZnTsDifjorq6ud1UXHce5yhK5PMrS5EnkcZJVU/Eoi81RWcmiEnFaqUKGVZyl4mEsHuNqLfJ6mcShWMV360rINBIrmSTiIY5UVvqDwe1aics/LsVShvfLLFU0JFKVCqusELJQYlVk31XqC3G7jktRFTJOS0EIfGqX+1XJyB98vfr4QaxlKaRYFpmM9KqAsUqEG1clY1OuVSFK+VSKx0KVVRKndwJAx2mY1JGKvKFIM8B18KCKeBWryBpW1svqKVcMknBSolB5BptPFYwXYSLjTdmMk2FYA4pP/sBxnMEg3tBQWdzlsijVAPa2EWGWJIp2ATjwgKusxr2Z8WGWP5lr2N46iZfm9u9lljKYXFb4woC4gVszqACiZhtzV8UbZa6/x/kqTlSDWfgwMZdpvcmfBBAzzXmB8j5Rskj9jaqKOGxwlbBteaeCvFBhXMIugjLMCjUU7QO4AKYHK3oRlHWOExt0siJcD3gJZmag0rsYREEPuKSH1/Rs2Gf6UHz97fcvV9fB118v4frm+mooovgO+DUYDG4uv1x/vsVXYiac8/DkfHQeLcejUynDaLQ6fr88m8iROjsdn50vJ5EKR+8v1EiOz05Po4uL0+Xy/P3Z8fJMHUdRtDwD/g0itQJpiCsVIOFdpPlQPMikVt50IOAPn8BiSH166zVPfWC5Sit/cx/Fhcs35ey2qGFT6ltcVkF2T7c8pVK4e1k8ATSajtoEtFut4m8E2edr8VY4CLqKZeL0pvqMaqW+VS7i60fA09IlfIcg7xGgMJvA8mlZA2NkGcbx7KNMSngLDMseg1Sm/MDDUWEWgRLMnLpaHZ1vLQZ6kMhQ6W0zrcqsLkJVuhuZxitgypDUPyiyDC7LcBXRpaYdqMj1N9JBUYFeoTUgY0F2pE7v0+wxRelFRUQzca9Ujpr2d9Ae0Hkkgo9qhrCK7LEEys0XdLcCWwKU2MCehUFl7mjknAUvj3/xisbNHZmCDZAkjWDbqhpGiRkIUapUVAZ607AR9RCrR6cFgH9hllZxWqttqJGsZKkqgPVvAOtuE4WlnBwHD5Ng7AiNI+CVJzGOQSsECD87ZPScoXAisDFJlm+Ab3hbwTacl+7iMLRU4g9k8XVRZIXr/J6qbzmQSGnby9vW7CP0UKVAKIBejIB5AJuejxbNOKSpL/NcpZH77JAJd6btNt45IIsMgN8thh3Mmj8HJQRmllXhkqI0MuGJd10I3j4QhpLW+kiRuyKrc3jIQPhuLxpM52mX6gCEjDs+12SY6yf7AaH360ygJ0GpQBSikoAC0Toj4N56v49Qxl0FEbgRlMYG3e1XLbgXr5H6VZZEqhhqHwjS5LoOeUWg1hiU2nXS7KO+H3leK0thEueoQSWYYBW5zKhWY4FRDNrz75Js6Tpv/E1+7HieLfSJSl2C46G8j09Gr4jqyiEP/syQX6aiEVyYyxiBAQHfGInnFvaLJcq4ZXyKO6W33RWXdXivKpTztHK1J/XLtZycnLour4r2lCQZp/up3CjPJ8unXM/z1+obexfXm0/Pga/jU0/8hxiPOsuQKKHLsfUVCWLWn4mRUGBVtQp3X43NK9L6LsV2ayCyhXDet4U98tVXRhzvDTvKhaBtxTq8VgmieWgto3H0a2ka/R6cqDVs5I8aXfqcYTywW0fwndYCLYdIOhLD45MLNLX47BkeNqaGZAceoOjg4Bca3UxtBWnbwn4C+djkCUStCDmqYXehhBsd+xo/w/gQJVGznl8aLbWWbde5V+j7XcKxcRxDQffasnW0DccDEA0fHSRfzuEFORyaqc3ca07jK7vVd7kqAEHwmSEEvUreQ6RnqZu9QBc+O2FV1QVvS8cDAGedFaXLJIFQTbNMkzc2EYPtKtHtkhhaLGDAczMdrNFk0YegjTZp29bUNPchoC9zDFg28ps78sctNuKN8Cde9/58KN57fpWBrIHyEzgSSoyLIrKTvCpLKvAJwhTXABg2r1F0F41goocfiZ9nDEr8jLAOCdpH9N+Nu87qqgTZhXQJzVPESRWoKGijWc64UYL57+JSlGsMqTkOqGL4H4OVIl7WYIggbQoVAgdYeVbGVfygIPiC3OERUp3PlN8QUTU0DNLqlJ7AjKUCQVHv5AoACAzckXAZsNzK+CidUHcSIfuDPexgWrwVLhL2iEnjEUtO9pqIvXPOToZiYvGNxXBVgFEsA0A4ILw4ZmXKhzIHlBSwFJIh/w98fcWPXLSRNFL7V1JjPd6/U5WLM64ub4KbL7/dBB9vvnYYbQbG5W9gwMGnemgt8M0YRQChwc94MjokAVcUmGqWa173sKK9tbHv4xoSPIGJRQs3ux/yOAv/ArIpt2NQcKHsvudBYdi9PYjAoA/N1Xy8EL+I0+Oem28WAuJAIh1/Vy49guADxqJBA4fu2nBGC2AdvMMgowPes+IL3qfxhnS3FXzwGA+wQuq+6RL3tWg5eRJJBqmGrVZdUrekA9NYKrfjcMzaP4v3hxh6m2VipR4bJeZpO5fTusIjhrglk2VJ9D5RgD7YtV4Ptb011hUSvm8sGmicbCSPBJg/soOGHy5PRZ2K4fUxatM5MOT9CK8AuM0LK/qKyZHJ9E65F96ip+Ogl+G9O+dV54QP+126xJkax0WjqUoiiUtXFnclOb8SKwtZlYVZoveFr/xQhmv1o8n1Rhb3YKVm1lTYmmPg+pgsO4aZPNgnMCUoLXpWyqaTTEaY2tJrVB/OtLfyZAokDOyD5p03e8QImRkiXCM5o5/gIstgigQb+ih4EEdh2iVblQnGyiIVDdCFlVmnpuI+KkwAyhkRQ9+A84PUNlT8kK+9Voq6qXVab5a4mA5jFNwrcHzKZXaNreDJro9QvEDh52LL7lCFIwYTDcbL9XYp7Ud48zmrPqK0Mv3aektrH4baSu+2+M3gxtPPOrr7jpTMDGqyER3q7Ig8F/Y+mvFxSXv6bOqbclk2oQHolhmHdmrsv2qhbsyy74zNaGBt4nIjq3DN5sOObrspktZt4JUJyGDsVjTWDUT76VIndoZ0wHn3zMCm/ukKcrJ9WdPkeNF1Dyz0gZaMjka6uDQWuNL8u9M1N1pO7NmNik63woSlvMOC1yqDFVgJYHCFdUT3VeO5ZeY6tbnd+OqC3A688Q8toXxQ3wNMG8C4lRAMNDCHiOwM/vmyxBqyC6NXYGqq9xPvQIpk/eUqgx3NYB6Eiu8nLu4bpJWegiQc2E9Tw7PJ2qM7qn+bg8KGId2C/yETM/moLRmdbLKXyhzYTJNv2tnOsE0g7VTjIByTYdIEU8YRDrMWnvPFS9f+kDmjtJ5T9cbE4e2OjJCIXmBNwfl4fXn7+5frr5AzG6sI+fGwnQWyldTl2nJGltXuuiMmNfkirtZ+M14RPU+A4FwarFHRNU/LN7m7YHkHfdXhnJRKsRisA/gGOJsClIRFq6ymgpzH4T0IIxeRMf6n6vO0X9aYOyDxmKzxeYCPxwEBHUFwuVq/9/EoxO2JMBtjLd66uKMnNUK/Ix/9T14JtAMyHshvgw2ZK46o8KSidNd0wsBeTPtAxhxf+AoWcTuQ56SoNM1tNwXZh6tnez4GWZ5fxnebLIZ42w/zur+ftmKhAXhcERr5oz5nFlog+AxK+1vaO/xWa9jDGgIEjTSJP4V/tjZsgzR5NCQ0PIUCmGcIDccvh8KXa1yYHRFXwKhYLpYZyEyYSDBzpgbSHA0BcelsCLQCAtAAzzoOnxq5zfqM2hClUZayKKSWFAz3Z+3eh+aAauYs4xRsHOjSd1VkQRQ/0DqzUYeFz45Eu8OM3HO25Zq19YI9u+w0wxtAzRMsr/HGmnd8iy9W4+bhatwHyol4idWy1pY4Jk8vA1TNrK6CXBUlrVzWG3fLNHY4T8VGDEjQk3a1qCcU3osWNA5Bg4a8HZEzGfQ6A1lHQQPe1Gn8j5qcWBHM/dHJLoYNhX9x0tVRpgIqi4Y2RCczS+RmGUlRTcHo7BJ5nlZ53hyJCZa+wjSFUad6qs4jehmEtpytWW0NsTnAA3/OR7SgCSTdU1KjPqFQcWznhDSmy8UOL4VzWVOwMH/4mMfjaiGNCczpVg+BFsu5BrZo8DGcbaN2XYTN+eQeDwYMAu0idELQRcdexH6zOBCduLyH7mR6tLDPHADX532mCWe+7LJDu23RX+OypHNCtjwIgTJq3LPOAVUlMRbpeUw7CWLH2Zw3/7D7hH24OuA9BC4HruKODCZzB+QtvM8zGBRwmO10xQbIcXAwQmsPvfdP5fJgZ1p7jN4bCrmJHnFzfdUHia7WjKUKmIO+LwCzSeYPcz8++zlYP/9bgXUWizRHD2PBSbxod6l9By4APOv1AbhY43O3klXy06TWeIqrwOhDxM9+n0KYV9iDNZE8SLKQvNrMAWft/FgMrmEG2KjCsd4QyzmwvBX5LSEQDgBdLB/bAcdeFRuKrR0SDFSlQ0BY1XbMpgfgNfL6hxsTjOnsJGwMYn8NpTNpbymlO+r/oqJyi5YM7cB2MaVuKika+X2llA5W/YpKlmNDwncqJrFc0RP/MpKbvxEnMBeEpBLkCnYMiUDBJZUEMgMWEpCAUD7N/NFYVxWx+QfNu+4CglS7ewDU2EePDSQaN8tkDyz5a7CqVFoCQeZj4BEvMN8PdrEL7sE8S0SYqs54rSirl4nq4vFuZi59jEu07FJHCP4fyHwo1iAVGSbAoKsJVqn4hO9ojId+c0Bx1B67UBUGO5n8TQbRS5bGodvmLhAahOu2EEkEp2dl19v0AZijA4D+Cwv2Jk7paOSNOB1hlbRfQNZp3+2Xy0+fxbKOwAZCCIOBQyRkxYiAGaHf7dRvd1WdpIajlXZYVkSWkG3qpIrTbBNDBtJU7DBaGk/OOdm0BMLDKJvyerQivHz3JIvi6QwD9LawZ2iZrVYgIS0xRwyf8AHI5708uFEIn6Lsu6JzqsAWqwrRcMxbFOfxoi0eE+Q5LzvVq78V54uuK0yyu9iSb11Z3pF52TZPp192XEHYLLoYNokSL+KDkQiMAu2LT3bCAZI2GKapv6pTci0y8TkXCcIChgTAlQLyWgrhA17T5R/duFFuw/WxY/JRFn3qNovVVZxADJvEOTEhSLNiE+yySBN/5O1hYVmp3N1e2zp2oXib8IkgDAjXrn0sAFTnhKn8Mx6O/FSORVeO+A/MbReElOIEkgDI5Vqx0falLV6xdk61uXiLZyAO7kUnT7xVrAKjwOs7Ds8DShJlbhWO+mWc1iWadXUlR992Sk6wv1+MIURE1NFx79ytayqNhcRyCBBL5VQXIeZagQ5qPo4e7ZAQLEC6z/dT8dBwjmsRJNT3kFKRVMOCPjYcgaS8DPu+vh/MtpXypOyVeBjhtxAJDnaZTiCL1aCoKTQ/Gi+83UYTyxMEEVL9830nlNhVg4FRXJIn6ccJt2D8YS86UvicWaEmOvmoDrH1F1QS3eBasSvEWILNvHMgwsRldY+ASqi2/2fDvCbhRqb3c/ADgOz1TcMmyNDrWrk3ZITh4HwoDx7Y/TnYIzkVjumEoa4hDiaD7UxlaqUpbVzRTR6mpI+G6oHpopnq1p1e4043WHmxklGT7eTg8UKsVju6L4eazexG8bfiv//64err5eRY3KkURDuh3oV3lx/+4tjwsAuODCulk0BMGBxghRXtCpUSTes4rMb9tO1sbiaH5AsSKxgTBmzhA/kgY7iEHP0H5pi+8mCjJPbu7lrIloSpeNbMwFr4q3bUJCZ2DXF/+Oek2AL7A2BtYbRBv1hYk9y9hq4Rzib5eQXR/RgaSJZWHMIsWD4F7XnGc2cl/bz/WDfZdTczL1qp3UaHija9Fih9s/iB9HP+wMC1Df8e54eJ5/3J9Q6Rvkv5/79t/+CeLTb/qV3v2/LLoF/f0xC5wme3KlNLW+vRtiRNN7YD8Y7RDJoF4b6tbbdTTB14dxm42Ffz7Vd8m3qvBZoTJ213TfDSscycMgVZSqWSJNhAOBGnuwwZhF5F9gBj8SA+UpusO+ZlcDiSYjEygZT2RDoRp0GBVdV7fvPG3AG2u3yQXaPbH9XslYU9TPmn3d5WsysEc0g8PukPG+dEfXT01dVP3U+SUL6QnaXCyL5SyZNoHND+ZfbzAgkNbMzXEDWRhdjlX/rg+r77NV5uFVaHfR7qIkGdRhTr9uZvNvprIQtQQF8SGG9JpPLBFHAZis4u9QdP/v/EOTaQuAwddvyI50Lm5aeb4MP1x79c3l5/oDPLHHIteWdFkNT1gm1tpNZbVcNdm+sJb69Q1j9N5gX5A54e5aihm/43BTk+ef70+dPXX68/dEPpZ0vwdZc3b9kbNr5W65Gxki+9mJtPTzZUjjAfORWlaZ+ia/+yuKvRy9/QGzdSZVjEORVNgyDKwiBoizMt3cynOEgO5NwRflpgbGNzo+sbeEknMnjBxHA6jUW4si+jKJAaGdc5OsKiPxOMClQ3VMIr1D/qGIImu6q5ezqYtjtcUB9CzVzHNKQ5xgI63j8DkCNpB1PVlQSqz5ywjuR05BycpU2w3gSwsp1/PDo4U9evdk69ODw1KcwsyvLbeSN/NDLFShRN+kKNQNAPAindTusrNbhaVThqdB2NTP/rSfPalNvg/nQ02uKu4lztv7DnC5KzXkbWsMF8msalMpnWYMxKpSL3eOL9hOeP/EGk3z7rPbCmg23FPgTKtrBGfQyjsZMVnn+uN7f66dj0FbJAd8+UeGf61Q+dHrXfqLU9ZlWh6MRGfzx2IpcyPDu9eD+ZXFyMRuHqfLQ6m0zU+dn5arkcjy8m4/EqHIfO6wcvTdID2xo3a2si6qPG7Q/5aFfW13x0334g1DkrQG/caPtu/2ve7oohuWNrax41EQbUIgj2hT+QRF13nc53pH7+5Hi7wJL4AGUPw90181908L1zt2nn21U+aZvyN6w7phJl1YPpK+iSe89XPm1YSKegGFNQMBGAxzdeEi6Ri2R2FTn648mLEUReBY0gfYbRGD/LEx5s0B1sV4MA6tZ5654OXO9fOAM62Fn7rto6DzItk1YovrNRANwgYB6QgAUBUSMI0CkGgaYGe8jB/wI5z+EhJT8AAA==')))
manifest_path = WORK / 'gmdcsa24_v2_1_manifest.json'
manifest_path.write_bytes(gzip.decompress(base64.b64decode('H4sIAAAAAAAC/+2dS29cV5Ln9wPMdyBqNQPU0CdeJyLKqyl012pWU4teFAzhPG1Ny5Ih0a6uadR3n3OZyUcyqbw3OZJumpkFl00mMyk+fop3/OM//+t/ubr6w6fyU/s5vfmtffz09sP7P/zpCv64efzDrx9LG+//4aebm18+/em77/5ve/+hfrj+8PHH7z628uFj/fQdEAmz0B82r3n4JH/4Da9h++gvv+Z3bz/91D6+ufnY2ptPP6XpCZJyKhqdEN1DKN1CV8Rmaj1nAEeAXqBsP0n6WH56+1t783OV6dVUKXYsUiDFGLN7JG/cvFdjaj1vX/Xpl3dvb9788uHd2/KP6WWffs3/p5WbT1fwP/Dq5mN6+/77K7qq7bf27sMvP7f3N99f8dVN+zT++/7DVU/v3l2V9O5t/phuxvd19andXP2j3dx9Xx8+vb2ZvqZUpo9+Gn/A36YPjA/1tz/+tH3aeO/vH8dnfNfu358+7x+mt3/YfKL37cf0mU/0/sPHn9O7+1f+9OuP92//OH6YD3/G+FLevHv78Gd8+uXDx/Gdv0v/ePwnlQ+/vr+ZPv9/bp/26HfTfh4vGX/Wn67U/rj34fLu7S9v7r7Q8RyDu+e8b61+un/5m4/tt7ft7xNH08f/uQXjbW0f3vS379qnibS3/W2r4ynjB/GpPcbt0be+/QrvXjz99v66+e1dwXd/GT/B7wJc//wL333H44k13aTxK5qe+uPPtXxKyG9+wzfw6Ck/fvzw6y+Pn/DdFomdZ20fu//LsP2JDpaml95y8/jZt1/6m5Ju2o8fPt5yNn19j57x8GOsv25QevNp/A16X6fvN16H555Z3qVP04/1P27uPuHb9z9e/be//tt//xtdD0g/XMUfvr/669ubm/H438L0wHj8h2f/1B8HHPndhzx9Jo5cE5Uk4LmF2J1aVTC3YBiAEidt3FN79hPlf9zc/o6iOUSXh6ek9+8/3Gy/s/HfX6cnPQfXwyvevr9pH38bADz8ynd+7Zsf7U0aFD/8rMZ3+MedJ7T39TM/ys2XdfsXavpa+s5v5PaD71Ju7+5A3fzvn3dv/rB5459/XAwjnimMcO3Pwjgen4exh0pE1YhLAuo1lJAcQ9OkhrEZhVw9kx2GkREJ7ZvDOL7DU4WRXgWMcjSMeC0Te/IUxvH4PIwaWnTG5mkYw96Vikmn3khN4rCYgS1mqnoYxkCBkL85jOM7PASjrAgjvwoY+QUw0sQe78OI8zCGyjLcM1SEbCOoDS45KWRM5ElG4NwwWqzpIIzsMZqtASMdgpFXhFHO1DLCtW0s45++v/q39O7fH/lpWxA0NhhhInKPBVPXEDFHzjC56um9ItB6c84HaRRTjCIr+Gk7VdMYz9ZP452f3oERr2Eextpybx2gJgg9tso5lGRczIOqIwUfnCLEGRgpBF3DNOKpwqhnaxrhWRgXoGgVFRgZIvRC0EkgFcJaQysG1kTzSGvksJeWoGaOK9hFOFUU7QxDxr8MFMPGS/NTFMMSJ928JMo4ZS2hUR2WUKxK7D2ylEowQdmxHU6mmZgC+zeHMRx20muGjH62MOp9/nKT3tdHNMZ5GpF6J1d2GmlKTJ5zEZJKQVI2G5SyNooZD9MoJiP7XoFGPVEaIVxofEqjLqAxVimlKnnPXiiHJrXXkVVnSMOJJ+bONFLsOdsYMcQLjQ80nmML5s9TzHhf895hcUHIqA4atIXOvY48Wj22DMMWinSili2wSE4zJW/xCMC8QsgYTrTkDXiuJD7qv+yyuCRmxAo+VbzZtaH1xmgIip2jaFPO3Di25DPdQHEn9UsD5oFGOkMv/edbL82f89I0T6NgF0c2j5RaNFJVwJHAcC4AMY2shtg0l4M0ApkHxBW8NJ+ql+YzrOwMGq8GjnTfD9zieHXH44ImTAsItZUIrVPyniKCtM7YWsTMxQ1GBGkAh3k00xh0BR7pRMs7IOc5K3F114WJP1zd9wS3PC5pCYYWMyMpNIbcDEc2olaCo1iXVrqOdBvwcA6DYQow46k1Ydb01fFcabzCOxz3aFwyvFNzCxRHMs1BpEFpWCMKEWatzSlqlmhWaGZeAqJDXKEPc4I8/s9/+V+nPNU4vrylNNoSGnfnxSYI37X2y/TIbXdwQfoSRn5iWXIcllG8U67QMknKvVJAqbFH82KHEbQAA1ZajODuKOz/t5c+mFHb5zncHQ1+QmL4EiTiayAxHk1iuJbHLMJmpmxBybtlzTAy6NJj7CH4yFpSK5JRJzSjhDQym5DksEGMOmLKcJI0xjVppAuNE40LWQRqWkOuWUoqyVVbbC6aOMXGI4QcOU1DgDRX2BE+YrL2bFjk18CiHMPi48naOxa3ZcYlo7WUU9aeIzpIStBQxwMlW5JC3Kxz19R9Zk5ixJTB4knSKGvSKK+BRghLcPzfLd3WFOMteuHRaC3dP3I3OXFrKhdUHQFHnoIGpcYcknMOPpLrgqKVuYVCUgF7Oey2YaRCfszg97ekE8KaeMZXgSceg+cGRtyb/F4y9x0FmnYzy2CROVMi0yqJWu/SeoNunA8PNwIy0ghATxNGXBNGfQ0w6jEs3pIn13s7MQumJiQMu5dpIAepJk1t/Mtb8lyFWPu0PWhV9PBGjOrIruNpJjS6Jop2NgnNzizjguQluNYkEdEySh2ut1SrZSp358Lj/zE1R+LDRjBSiKrxkrw85c5fA3d8tAncG6pdsCwNNaFz7Np6ADMeCMbKlkvpwxNnL5yzmBwODdmI1E8TRF4RxBMeZTwCRDoaRLzeWzVYkKUUz7EOCptDSjF2GTjmTIFLCsULtJANKR6ejSBjBT3Ngg6tieKraLr4MSjelnP8aQLtextZ82RSk2bcmwwzN1LmWBpMfhoD9QrqUAaeHCgcJNPR3Q1Pkkxfk0w8z4QF9xMWuqbHY2V0P9tzB6xs0prv5916bMONM4wcRtK0l5BGstxGqMk9RpZp2gxCKnUmrwGBQJe85imx59eoobspiifLrAsq46whWoVQOkYaiU0s0t2F02RJe/FJAMW99cOpDoIHv6Q6eyieX5/mYZbizk7yZnHmccwJ17f1c16yOUMldcmBTVpOTDWBTpprueeaUzMQdcE6Ux0fOVII0S+tm6eAytl49ztAeeukn6pQLLCVHgZ+jixAWLqGAgGQW532aKS42HgMSz68OqNIFNQvbvspivFssvHdqHJJOXLk3DRtxLQRHVofPjv5yHDAqwfo4JiHYdTD3WsSQ0S5pN733OHXkHHEZ8HD01TO4/3Nrc1jsGQ7IXYKNjUHqYpGa8kgO2FlSYLQZCTg0cRxRjxPUXENUZ6TU3LEr6HkuBqP9gJRHt2M3e7lMUvmzQpYTthr4Tygy63VGMQCsIReLDobQG6Hi0DjhQLoKwyD6wuHcL86jPQqYNTj1/0XtqgT9DTNSQBg6la6eZdhFCtPNXEaSYdUiVjzTEwYUDyssKL10nDwq3PHr4K7ZUNlOxus27LiZrZsZ4WVloiTeeVkOUDqzgglShgOmRpISCA1YgktjcSZZobJZCTWhivoK9NL58i+OpHyKog8Xkb0kSbUC4RPqNnIk2viDNqmhSzl0DxH7plqERWqEPNMfZHHi9guolCPaIxnmLT8+TZIhGeL3Yvkvl3JKBVVqwFZvTYqBbnmAaT3Hko0djGeWZDhEFdRboRTzVj0DE3jFCTi84YRlhhGzVD6CBNryAScSapNEST1XGqPViJS7YFm9MkURsYiK7AYTtUw2pkaRtpI2k6G8UkjG/Ya2QuERUvV6boNTok0YNEWYkolcGLpdVKL4tTRZg4jIIc1tBzpsMbtmpbSXwWd8gKVKHte43aJqSxpxI4VR/xotaExS261VundoMdAveWEVspMC5CNYryof9/D+GXnH9fLseH4IPLOb+8kOpvKN/zwEoH63nU48Nx1GM4O2bUQSakpZ50kb3NQKGY6s3odgipfvPkjROE8y0B/k42q3s4+4ZbGJTpmWUNvneKwjoYegmmC7hHzwJCxDM5sePcZhXqdhMW/fQ1IDqvqrVkDgtfRmlm2S7hjMeOWx2ml8F8+vn3/75sJyM1O9tO5yNvhnvGKvSnz60UiFkUIpTrklL2Y8yQy1RoEpTR+mTnmUC0cbuQgOdEKqj5xhl1ckV06w7binx9uctlemr5oPrL2Ytkco/nk4XEynZiwkkcKXnvTWIDyYRwhgsoRo7rf6CjXin1F4HONPbfHMwGe6e/gkouFJVvoHEropDVQS6WJxjpSbbHh6C0l9JllB4QRZkY9tfOZACsCKecI5JT26GbsAvbKROMDkwf/+PbhIV40ihFrEddagpfcp1mg1DA19pyHQ3dNBu4tz6iWTkKRFuGbM6qHZzFWZTSeaZ3982dpFuA4cnJWTV2m9RqdThhOo0DmlVuTkeSYYquhzRTaMVgMcOlAPsCoZ1ho/8tDPBlfdDuuTGXM0IOO1Np6bt661aBZshWU1lOzkfu0mTqRCms8uXByxbo62Dl674lG35jGW+/9KDH3TWL+XNJDj8C9OxG7pKhUPTBh8Uq9lsIBa68owYS9s2UEqdMyxIw4PvgwpPHbH8v2w4Z0Vbf+OppCIEfDa5ueJUxtoX9NmzBzoDtFmU+5tSXWNcbCsdYepm4RNwbs3ZmAVUomNAlFQouHu5YgPmzzCsdt7HDbElZsFWE4V/uq28InPCC63V6c0vWnlOqikbiKubc0khuDxJIqGGWv2Um5uuTpxMME7IwhVRcGXiE/8lM1pPg6ukV+dAofN3bU93jknSI8b081LZsjJo49eBJpYrmhTusTHC2WEb6mDBqik/thUY0RqNoKdc942JL6iojiGQ6A/OV2AIQ+MwCCS+4ySQhdJj2CbFyns9vDeIYsXGLrRDYtW4DOnH8f4cAAcoUBEDrRARA8x+Wev97W4OPdFvjTEvySy7K1lW5IXkNuSau1VgP1QaJ6hwrjYyGVRnOqlBIcVijBxxNd+UE+Wxo3+i3X/LQCD9c7iOLiLpFUYMl9UFi7dHBusWoWytTYZcSe0w3a8ZzDiJK5iJ3aElB82kT6hojKGSJ627T8jLlcUPA0CthjsTwdf4BmsZpBwYpA1ZC5CZDi4aqRigTWNUg8wf3IL3+k6YuCeIymORyhlnF1t5L7+JSibkynXT+e2rTNihAsyHJ6096i1j6SmUqDyNDSyMrRC0aTzINSLFHmUvGIyH6aqpaH8vGvqq3x5S84nTam98V3CJsG0eOG+hbe6aFFTBYK3VuDnIk6hxin4bdachw+m6yn7L3NMGkSI/qFyT0m6exM599km+9sIsy78zn2/Lz7ohkP1tq4xoQ5YPaRk+eSJiylJM+NmpuEajPba4Cgkzb1BdI9SPk1QKqLxt7/9T/ax/L206O7EHtV9QU19RSjTAKq7CX2PmLKXkeUGVpkCFZEA/lIyQ8rqo5QNYifpkSWhjVxlFdhM8PR4my420GnTWTJO+Emb8PNsIBSah3a8OyTIJFiyKVP5xkrJZ0eMYBhR31mqxIAFMH0culpD9N4fpj6ZnqTnlHnkO8HqZul4F3F/218em3fX8VN+rRkX92aBcnUYskjS4rFAxUcaRKkLL3EON4DSTzDLhFFwgu7e+zqq2CXXuLxYZ9eWCKLmaCZKeai0/ymxpRD4E6RjVuGrlTB0Q77/AnI8fzTdPpAaxJpFyLvFKw3E5/X+P0wEkvx7DpdigqerUoYKZGO99Gg56SunKl2ocL5cAcdOARA0Quee3j6mcakt46dtksbuypxd46d9r39IvG4xEKhAwm4e7AmraJ3BMgVgmYKVWMuMld6QlIGufj4p8iesODCF1dbf2ASHlRoHsaOYTMMMm9EJZSRJOXUiUISQEPWnJKhFeXITphKSTPymopuerkAuccjnJEJfTIX9/gCwNa7D7NqSzN56FCm0+GMFCVHi1JM0/DwHmppYsN+xoQzkuwQgDngJRvaJxPPx1LeleM/02MPixx3MioFRMSjaePp4FnqqToGZ89Nc+spwtxBKYeLmXwGxlfRMTrqYCncCxtuI0/4nATsAh/ePA70YsfeAkYgS55ijy0k8lxT1dJireXwPrAExlOFk9eEk8+nnXlHp23amTsb6/zoseNFZzxA0yrSkw8PHg3INeh0aRc5C+hw0zKYPVyYH7YXglxa7vuMyvkxGveHQO5nQOajS/I4aBSBHFCS2qSgTdiDewtCwxImK9h57rKKEcMFxz0c4/nhGB7tA98FnL4Ux6wQmeKkwOWkkBStoqBzq6BQmhZM0ePhstC0sCHGFxz3cNQzGkh66KTfmkf+fuTdtqd0uG2pxyXKm6Y8rKSRYBw5DoBB66jTGd0qnGrWkpJ4mrlCReh2opMf68JpZwgnPh9JLti7aHWYyQImBTIQDGPZC902e5C1lSnEjC3GPCObTRKILjDuwejnN7oJz7MYbiUS7rS5ZKknb2mqN/aRpdSYK3OtLZBAQ45dasutdMiRy4xMsbvYJc/Z4xPDGY6760bP0He21+ixptzxIodUPPRQdISYNH4kGpoJJlBMNRaWkQdlSS3rjBkVF7/kP/uYnm3bJ14/Vu2K25LRgnZPbKhYexfJoWVPZbqKUYMCWwzKVFtrklOaEdQetjZcuj37QL6ONSE8ctIINnIctpHPvp/n3DjzRZX1WrloFI7uKCPXaSVOxUtrDrVJ8UINe81zuY8744lur+GaWNLZNCG3ykbbcWHZKBvtaMEu6EFKZnDtU9sxN64l5YgoDCoaKSqVMixom5t3wxhPNLZcpwdJX+NKOT0LIz0DY22/tXcffvm5vb/5cgvotHQB/dEt3gUTl5GMvVtRjXkSGdYktUPs3jyEXF2bI3A+XKdkIBH3U7vFSyvsmtPXOEi+MnrxOPRwe8zicR6zlTLCWyt5/HGWWslRuYgViQljKMVZPWDrmDtxBsIRKB62k8PjO6xw4AIPy8fEFSmlV0TpSw6syZ2+1tNjlAsCSYaaUmuDypbDyLZrJhwoyogqRYsGHiFiD1lm8huhGHwFgS05NYEt+hpXzH+HTOLLNd9KY7fiMdQqaHm8k2XScM2aWFgyZMFG4XAPXKIrs62AJJ4qkvKKkDz6rtr9mdQnixMLA0yII5EBajRJuIdCdbxZugZpXXRYS6SR4hDQzNQvhqgrHFabOYu6ymE1+hrnzH9vhnKrDrNkHWIkMUYCYJ266aTCri27VbE+7GTpnLQAzNQhnafe5KllOGsaRX1F+Pnxfvp58eAlk0El8fDJI2lx1+Fss+nInRM3FLNYiBHVazo8N+kY0S2s4KTDqakE09c4Yv57ybjvBTE3vZoFIeJUVey9W86tCmXD8a51waS1asRgmLynmbk0QQbhU7OGa2bSfrb0/fmRHOuwhzvTk7TTp9k+xAuCxjAtLVbC1KgnRI/NudZUkFqaTpIHrUXjTDObKYSIp6fFuhqlX3aZdmVK+QV66rpZyZkfkAx9Oi7RteeY2ggaIw7gWuqT7cTUxxsiOHNQl4VGyEgr+Gg9tftn9DVOk6/Mnx3NH2/4s0eXezYzkvsnKJbs3xBGRquRm2YJQsEsde6hS42UUXuU8c9hT+6gIxT99gL/fJhRW5FRPFsbuak/xs+tLS4YLG+WErVQpITiw2P32B1KvT1+UgeizdrIdDDP2E0gNV3BbsZTtZt05kzy093abU2cnx7bXdZNRInkxWOszdFy60Wx5d60BKzWWwoKmWcoZQ+ua1DKp0rpa+rcLJtM28H0TkMdd3Ww4r3a+vGSwaTCI9d2xGyx88jd0aVDl5jFTVyS9qSkM5LBg3YyW+E4H7x0UO2rsypn3mX0z3W+F3j5nLpnhqxghjlRcXCrHcVbD/lW8VJqscPiBAQiHNewn36iFXV4TQ2d40/6wGapUfc730sGKWOc/LfklptFJouSezTpWXk8oB0p1TgzSEnE7iGugKSd2skU+hoHy39XCfujwNOeljXhGp6LReeL76Ex8LCT0xW+YUNDiam25C61eyEf78A06HY48BxYB4unFniumbK/ptbPkfd2/3pbV8LndTMWLY5xVxg5e8vVbpN2w1irOGGNjtXBqHsavn5mIQI4QlihjoQnd1mXvsaJ8t9diImfCzB9AZJTHVPMaqg9Azhr0zRgTJRKZU4GPUSaS3tgpD0rtH/wZEc2MJxtk3Izl37fpNzVKFhiJXGk4Vg4EY6YspQRVhYsbpk6skguolOxvcwd79FJaH0FJE+1I4lwtlNEk+eW7SL4QPJ+OfxRafNpV2hBiCkp1QQlFy1oPSXpbWTmTEyxkjTsMIwn4mFVS6cYI357TuVwiLnOdNGXvx35FTA9YrnRjlZPt6dV9gU1IuHUQrGC3EdOQxCNR3ZTG6ek1tSmfUdwP2wvzUEITlOOwFbabPzyNyLXxdFfcmtif/ByQVSZYKTZKVck9ZHplIAlS2aNpVCGkDmgQfE+YxotYjhNJH1NJOnMkLyXbXnmSuR2FPgRo3rXBuKdo7uyEF000lZyxpaMR/SZ8zCitfTeDFStDE+vOc/tiQcIiMAXdp+yy6+H3WMvoT2z2YOboJSu46Q0uL2C6g/Y0uYxmU776KaR5EtSqKgVsfXcSEggQZMo3rMDppyYxQfjCjxzKSVQCHiiotYrKcLQVzg4uS7F8pKgYL/YtGA7yPvw5tqrp4Y5hWBhGNrmRCOdN6pg3ZKHmUljmQ6m2mkGBbImkvHM0qY9JHcaRbIZEFkyRxejMAf1XEIY2bh3KByiZaBpZmkSaw2xzWXxwTiGS/a0R6W+HipfdKnvucOn9P3VVg5hUTnUJcTCBUOQ4pWMQgyS3EeWn5m4AToAhsNLlSTuhKcp2kprAmqvB1B+wa2+HcX1sBkLWVD6TGS1JE6tYkKtToGEc2tZ67CgACNFslplRpADJgVNu9xMecqkvx4m9RgmcXvIdC+/32gZ3dvRuEmlFswqlZ4MkyZQEcZUmuZuUl1TJB/uftIfDMSHk3kLYB7jRfTtCacnv+L2xdUxn/TXl8jEQORGRTG3ZCPSJMfMIFO7qFAT9Nhb6MXneplmJnDRwNxDEM4WQdxqWON+j32BYZTYpPY8yAyNs4ScC/WsHXgSevOem0O01Ge4pGmJHS9c7nGJ51bm3A5tPpqK2wSTdr1TnN8ubviOOOFDaXR+XEljsULYscdpgs5ZOdYRSQqM2HPk8tEGxxQPB54GzHipa+5jS+fde58XS5AAYFq6OdfMTVQSi09ZUOqS+5Sm19YP5z080iXVS7Fojz4+M/ru55DoUeJzJJFdCxhnL9KGYczDMCaUMt4bCTjF6QKaKuU8s2DpYbKgFyKfEilnlonvrF7oC1TWe+/Z3bH0hoIQMWUPar1FtzqS7tKD9zQzi8QQwyXhfgbHeG5R5WPtjk3nfGcRaEHAWNUVQkQcKTizVy2haB5uu0IJPRI49pYrz3TCRSJGukSMe0jquSXg9057u0TJzyyoLTnFN1DMfZBZSK155GgQunDnMHy2GFpO0WdOOkceptbCJf/ew/IVtXWWbaltLeW27709pXufisfPnelbMLYRkdxK0BKhEqVgJQEl6EQ1JZXCjgkaHC5hIijgqR6XgjVJfUXNnkV7QncAbs3nEgCH71U368bFJmG4UiBUwqzKLRfOAVrr9XCKA6RyzNbktwQwrsgfhjMdJl46yi5thI3SZTjaQVqItXLIXgtxt16qhwg59nLYUY8UHewIBY5zGQdGOLeMZnf3bHdo7WpxVsOTxAaJ5giKEYiKN2Fln4pAVAMTxmLR5rKa8Ty8ZDX7WOJ5G8UnO2gLrGRI3WPupQmKW0bD2tW5jvTaYwXJQWqv8bCygasAsFys5D2O/DWu6/GzPPJzpx7bp9XUCB/pFPHLrkaZRimx94YFY5OSckl9JNYtJKoykm7FpD3PnENRYA12akJFa2gP8te4t7cWjPgCGOVeQnh3+2zJsjjKSFaw19q5Vq21dcg947RzW3VkNaEnDqnNJDEAAcRO7YQZrggjvQYY40v11eOLFNwoaCwZK4RCrfeRTVMvEmCaJp/OkOaK2ooe7mIjI6xxmGJGcj2uyCKfn5f+y8MdvSVGMCGGRgaJR0JdCCQilVC4dxoxo09nR1FlRpTNdLr4eGpH89b0yPIawDtexnI7ArnXp8YlDtlr7i2H6bht1Mo9EcOIFL1ND6YaBosRAx++O6HRBFlWUBnSU5Ox5K9xKu/34ZA30SE8O8Kz4EpZHX43OmHJ2pLVEMgphwYR0RJq6CwxSk8zqoDKGGQFqwin6o71PEncago8ExouUEEXqVa71I7QXJR46vE5Qe/s2EMlpjTejjNtaSAXX8Eq8qmyaOfH4p93dQCfeOgluzPRg2LjllQhWgPN5sMo9ls1FrDqIUXWmREJIdEVtM9PUAeQv8b9vN8Xi/sgLpjUaSl3Z0m1CkNsGoFzQ8PsNh6wZBhjohkMkact7QuG9xh+2e3BtTCUF0ij3sv17kzb4v4c2fTQfDHHphslraiREnlzSuwNWLoXSKZ9Sq7JZg7RUyBkWAHPkztEz1/jft7vsJjzNHpcUN3pWhqMbCZLne7nyIgYk5NbCjLyZGOqIQoc7gEOf26RLtWdRyTiuZIonzlFtuDumIfWC0gEHElxji0lGPkxBgx92McRFHZRzTM5NZuAGZ9au2VNFun8nPbtye9NHrOg6Rzc+tQ8wR5HVKhee6mMUYL0UibJPQKySofLioKAEHCFo990qu6YzxU8u9fbe6rcMz8lVnLFqLFOWmYhB0RuuWOTSlwBNKQUzeqc4J5PV0JXYNFOlUU5S4e854aXrANWtxwIMiUVa101lWjUJNbOMUHK7iN9mUlOwnQ6eW50+31r9dM9fW8+tt/etr9/DsLjf+fx/FpsUxBGd1JLeyIjC4YCIaUcrVqj0lLJ3RhE8vCIVNliqSVESwkPV06UwshNv/25IzqcEKzYYoOzbWzYZ4rJcUlyigiRak4ZUhUcJqU0g1Ra85A1kvVs7IcbGwojvCNboUxip1fF+/LHOr4kil9eeOnYDU+tuQeLPqxeo95qMOA+yXi1nIbtS4DR3GOdkVgyQzmiHnIGK578FQ5zrIVeOGZlbvF6SAAfCSRmH9YOgQdnoVTlILEFJInFi00iCzPDpsGms8OX9ZA99OgVoGfH6IHQVsxrb1lpQRZKeZAkI/dkxJ4xl8bYQwbnkQRUVSq9JpCZWzBMw5biRaHmKYr8GqwgHe+AF9SAVTMwjH+Y2AK0YDmrAg2bmKbRAgVksTpjBSkqeThRB0xroievAD09UtQdt0da9oerltxadchCWrlySzCY1MFWQR9Y9txarCXHXPrhepy6GpzooTZdE8f4CnCMR+IIn5v1W7Ad1zy17twsBO+NamrDIKZQSaFLzl5bEWk4kxDryZ5oi2vCqK8ARn7JwYsFk/eWYkp1asX22DVM8h2ZWqkumKUVZmkheTicGIuoRLwcD9gjz86mInMvpbA97ncrhf1wSwAeHnuyrrnAOjbLnlCzSDfq2JAaE0Qv0EzQVRLAsJBzN6hdRrZ90W3fp9TPhtKdIb8dRBdXEjEpsyswszYa/tZ68y6laWwpk+aAjo3mxNrHq2O4VBL3YDzdSdQvX8S+s4S8GWjZvx6waJm9VnUcSUjqMTI2j9UI8vDataSWEGpP3XhG5RUITelEb6SuCyScTWn7qZ7rg/LM3X2LBeVuCarU1Ie/rQalNUPySuRYEKOZZzPDmubK3eASLlcB9nHEcyl330eUvIko7emQ/rRDt6fR/kQtadGeXSqMip5DrZaGVy/TNZZEU2WyAEbqXmscQebhw1Sm7uqXovhTYF9Df0aONp/ylERZtD0SR4aTGHGEkxbqyKsxESpxotBqhGE3Mx/WSBLlkQ3x5SbvUw75XA1nvNaDh0/tcdi5qWEuCjw5lqwGKUnvUKVrcRpO31H6iAFS92IwPnK4pWOMElAvZvMprnKWZnMSwX5UI4JNh2eJLnGtwXGk1MoZxEqTSSBWIwOKNq09BBF2n6ljQkS52M49GOO5dBcfVNq3N6F3bvLitS88vYI0SAy1eWbwiIlDCKU4cIswad3UApJhZvNEyeSYaf9z6S6Cnkt3cacatCCTqUUgtRKtWcBpkLa7d9fhZDmmxOyF2qROPKMYgggaL33Ep9i9im4OH1Mnh22ICDuLn9ti0IIQMYycJXtBzn0YxDDSGG4SxTiyp9SsoMUgOc2tnDCeaEYDa7YX4TU0bvyIgyq6Ic8fwRg3LtkfZzgbmZtrfpEeZ8ohBa+lIVIZtEKOOQ23PQj0WnFEkcWy0OFyOo0s3E/0RJWvSCy+iu6OHpmFbw7+hFsgtySPiHpzZld32uT28Nh9ur5hftFeDceeakeKVEqNvfQcCiXMXEbmE2NyVtKuPCNeR2QnOswBXyPsnP4z3v7n/wOeokn8ZJ4BAA==')))
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert len(manifest['sources']) == 160 and manifest['counts']['publisher_temporal'] == 78
print('GMDCSA metadata:', manifest['counts'], 'source revision:', SOURCE_REVISION)


In [ ]:
# Dùng gói checkpoint v1 trên Drive nếu có; nếu không Colab sẽ yêu cầu upload gói 2 MB.
parent_candidates = [
    DRIVE / 'artifacts/fall_general_v1/mmaction_interaction_v1_parent.zip',
    DRIVE / 'artifacts/mmaction_interaction_v1/mmaction_interaction_pilot.zip',
    DRIVE / 'artifacts/mmaction_indoor_review_v2/mmaction_interaction_pilot.zip',
]
parent_zip = next((p for p in parent_candidates if p.is_file()), None)
if parent_zip is None:
    print('Upload file mmaction_interaction_v1_parent.zip đi kèm notebook.')
    uploaded = files.upload()
    assert len(uploaded) == 1
    parent_zip = WORK / next(iter(uploaded))
    parent_zip.write_bytes(next(iter(uploaded.values())))
with zipfile.ZipFile(parent_zip) as package:
    assert {'interaction.pt', 'interaction.json'} <= set(package.namelist())
    meta = json.loads(package.read('interaction.json'))
    payload = package.read('interaction.pt')
    assert hashlib.sha256(payload).hexdigest() == meta['checkpoint_sha256'] == '8c5808db106aacd0f43b72a0e76178b2dec039e0a1766d996bb8374b7e4dddb7'
    (WEIGHTS / 'interaction.pt').write_bytes(payload)
    (WEIGHTS / 'interaction.json').write_bytes(package.read('interaction.json'))
print('Parent v1 verified:', meta['checkpoint_sha256'])


In [ ]:
# Tải trên Colab. Cache ở /content; không cần tải trước trên máy Windows.
fall_zip = WORK / 'gmdcsa24_v2.1.zip'
fall_url = 'https://zenodo.org/api/records/13354453/files/ekramalam/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-v2.1.zip/content'
subprocess.run(['curl', '-L', '--fail', '--retry', '5', '--retry-delay', '5', '-C', '-', '-o', str(fall_zip), fall_url], check=True)
assert fall_zip.stat().st_size == 1107545615, 'GMDCSA24 ZIP size mismatch'
md5 = hashlib.md5()
with fall_zip.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024*1024), b''):
        md5.update(chunk)
assert md5.hexdigest() == manifest['archive_md5'] == '3d36f2c5c1a666b99639e4e9fd843efb'
FALL_EXTRACT = WORK / 'fall'; FALL_EXTRACT.mkdir(exist_ok=True)
with zipfile.ZipFile(fall_zip) as package:
    assert sum(item.file_size for item in package.infolist()) < 5_000_000_000
    for item in package.infolist():
        assert (FALL_EXTRACT / item.filename).resolve().is_relative_to(FALL_EXTRACT.resolve()), item.filename
        assert ((item.external_attr >> 16) & 0o170000) != 0o120000, 'Symlink in archive'
    package.extractall(FALL_EXTRACT)
first = list(FALL_EXTRACT.rglob('Subject 1/Fall/01.mp4'))
assert len(first) == 1
FALL_ROOT = first[0].parents[2]
assert (FALL_ROOT / 'Subject 4/ADL/01.mp4').is_file()

SCFD_REVISION = 'ff83b7c521e5ca4eb67a212cc42dbc54f39d6290'
SCFD_ROOT = WORK / SCFD_REVISION
if not SCFD_ROOT.is_dir():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/seymanurakti/fight-detection-surv-dataset.git', str(SCFD_ROOT)], check=True)
    subprocess.run(['git', '-C', str(SCFD_ROOT), 'checkout', '--quiet', SCFD_REVISION], check=True)
assert subprocess.check_output(['git','-C',str(SCFD_ROOT),'rev-parse','HEAD'],text=True).strip() == SCFD_REVISION
assert len(list((SCFD_ROOT/'fight').glob('*.mp4'))) == 150
assert len(list((SCFD_ROOT/'noFight').glob('*.mp4'))) == 150
print('Data ready on Colab:', FALL_ROOT, SCFD_ROOT)


In [ ]:
# Môi trường riêng tương thích MMAction2 v1.2.0 và checkpoint v1.
os.environ.update(MPLBACKEND='Agg', CUDA_MODULE_LOADING='LAZY', OMP_NUM_THREADS='4')
ENV = WORK / 'env'; PY = ENV / 'bin/python'
def run(command, label, timeout=1800):
    log = RUN / (label + '.log')
    with log.open('a', encoding='utf-8') as output:
        result = subprocess.run([str(x) for x in command], stdout=output, stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        raise RuntimeError(f'{label} failed; see {log}')
    print(label, 'OK')
if not PY.is_file():
    run([sys.executable,'-m','pip','install','-q','uv==0.8.22'], 'install_uv')
    run(['uv','python','install','3.10'], 'install_python')
    run(['uv','venv','--python','3.10','--seed',ENV], 'create_env')
run([PY,'-m','pip','install','-q','numpy==1.26.4','setuptools==69.5.1','wheel'], 'install_base')
run([PY,'-m','pip','install','-q','torch==2.0.1+cu118','torchvision==0.15.2+cu118',
     '--index-url','https://download.pytorch.org/whl/cu118'], 'install_torch')
run([PY,'-m','pip','install','-q','mmcv==2.1.0','--only-binary=mmcv',
     '-f','https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html',
     'mmengine==0.10.7','mmdet==3.2.0','decord==0.6.0','einops==0.8.1',
     'importlib-metadata','scikit-learn==1.3.2','opencv-python==4.8.1.78'], 'install_openmmlab')
if not (DEMO / 'repo/.git').is_dir():
    run(['git','clone','--depth','1','--branch','v1.2.0','https://github.com/open-mmlab/mmaction2.git',DEMO/'repo'], 'clone_mmaction')
run([PY,'-m','pip','install','--no-deps','--no-build-isolation','-e',DEMO/'repo'], 'install_mmaction')
os.environ['PYTHONPATH'] = os.pathsep.join(map(str,(CODE,DEMO,DEMO/'repo')))
run([PY,'-c','import torch, mmcv; from mmcv.ops import roi_align; assert torch.cuda.is_available()'], 'check_cuda')


In [ ]:
# Tải trọng số gốc AVA + detector; dùng cache trên Drive nếu đã có.
WEIGHT_CACHE = DRIVE / 'artifacts/mmaction_interaction_fall_wrestle/public_weights'
WEIGHT_CACHE.mkdir(parents=True, exist_ok=True)
checks = {
 'action_slowonly_r101.pth': (
  'https://download.openmmlab.com/mmaction/detection/ava/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb_20201217-16378594.pth',
  '163785949c5178484012a6593d12d6fcf4d36be404fa42b5899b314e91f4703b'),
 'person_faster_rcnn_r50.pth': (
  'https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth',
  'a5d8aa15c75b961b9c73f28a6e92af370835639a2939ff890852b052a5256a5')}
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
for name,(url,expected) in checks.items():
    target = WEIGHTS/name; cached = WEIGHT_CACHE/name
    if not cached.is_file() or sha256(cached)!=expected:
        legacy_candidates = [
            DRIVE/'artifacts/mmaction_interaction_v1/public_weights'/name,
            DRIVE/'artifacts/mmaction_indoor_review_v2/public_weights'/name,
        ]
        legacy = next((p for p in legacy_candidates if p.is_file() and sha256(p)==expected), None)
        if legacy is not None:
            shutil.copyfile(legacy,cached)
        else:
            temporary=cached.with_suffix('.partial')
            verified=False
            for attempt in range(1,3):
                temporary.unlink(missing_ok=True)
                request=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0','Accept-Encoding':'identity'})
                with urllib.request.urlopen(request,timeout=180) as response,temporary.open('wb') as f:
                    shutil.copyfileobj(response,f)
                actual=sha256(temporary)
                if actual==expected:
                    temporary.replace(cached)
                    verified=True
                    break
                print(f'{name}: download {attempt}/2 got {temporary.stat().st_size} bytes, SHA-256 {actual}; expected {expected}')
                temporary.unlink(missing_ok=True)
            if not verified:
                print(f'Please upload the official {name} file (SHA-256 {expected}).')
                uploaded=files.upload()
                if len(uploaded)!=1:
                    raise ValueError('Upload exactly one official weight file')
                payload=next(iter(uploaded.values()))
                actual=hashlib.sha256(payload).hexdigest()
                if actual!=expected:
                    raise ValueError(f'Uploaded {name} has SHA-256 {actual}, expected {expected}')
                cached.write_bytes(payload)
    shutil.copyfile(cached,target)
print('AVA and person detector weights verified')


In [ ]:
# Feature extraction is resumable through Drive cache. No training on your computer.
base=[PY,DEMO/'train_general_fall_v1.py','--manifest',manifest_path,
      '--fall-root',FALL_ROOT,'--scfd-root',SCFD_ROOT,
      '--weights',WEIGHTS,'--cache',CACHE,'--output',RUN,
      '--device','cuda:0','--epochs',EPOCHS,'--minutes',TRAIN_MINUTES]
run(base+['--stage','features'], 'extract_features', timeout=8*3600)
run(base+['--stage','train'], 'fine_tune', timeout=(TRAIN_MINUTES+10)*60)


In [ ]:
report_path = RUN/'report.json'
assert report_path.is_file(), 'Training did not finish; inspect logs on Drive'
result = json.loads(report_path.read_text(encoding='utf-8'))
assert result['status']=='completed'
package=RUN/'mmaction_interaction_fall_wrestle_pilot.zip'
assert package.is_file()
print('CHECKPOINT:',RUN/'interaction.pt')
print('PACKAGE:',package)
print('REPORT:',report_path)
print('Parent vs new on GMDCSA24 test:',json.dumps(result['test_by_dataset']['gmdcsa24'],indent=2))
print('Fight test:',json.dumps(result['test_by_dataset']['scfd'],indent=2))
print('Wrestling separate metric available:',result['wrestle_specific_accuracy_measured'])
print('Checkpoint has not replaced the app model. Review report before installing it.')

# Tùy chọn: bỏ chú thích để tải gói checkpoint ngay về trình duyệt.
# files.download(str(package))
